In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
R = np.array([
    [3, 0],
    [4, 0]
])

m = R.shape[1]

r_bar = R @ np.ones(m) / m
X = R - r_bar[:, None]

print("Mean vector:")
print(r_bar)

print("\nCentered matrix:")
print(X)

print("\nRow sums:")
print(X.sum(axis=1))

Mean vector:
[1.5 2. ]

Centered matrix:
[[ 1.5 -1.5]
 [ 2.  -2. ]]

Row sums:
[0. 0.]


In [5]:
#import from lecture 3
data = pd.read_csv("market_data_2007_2025.csv")
data["Date"] = pd.to_datetime(data["Date"])
data = data.set_index("Date")

prices = data[[
    "SPY_Adj_Close",
    "QQQ_Adj_Close",
    "TLT_Adj_Close",
    "HYG_Adj_Close",
    "VIX_Close"
]]
prices.columns = ["SPY", "QQQ", "TLT", "HYG", "VIX"]

log_changes = np.log(prices).diff().dropna()

R = log_changes.T.to_numpy()
dates = log_changes.index

In [6]:
print("Shape:", R.shape)
print("First date:", dates[0])
print("Last date:", dates[-1])
print("Row order: SPY, QQQ, TLT, HYG, VIX")

Shape: (5, 4712)
First date: 2007-04-12 00:00:00
Last date: 2025-12-31 00:00:00
Row order: SPY, QQQ, TLT, HYG, VIX


In [7]:
m = R.shape[1]

r_bar = R @ np.ones(m) / m
X = R - r_bar[:, None]

In [8]:
s = np.sqrt(np.mean(X**2, axis=1))
print(s)

[0.01250852 0.01408005 0.00961727 0.00692031 0.07714686]


In [9]:
Z = X / s[:, None]

In [10]:
row_means = np.mean(Z, axis=1)
row_variances = np.var(Z, axis=1)

print("Row means:", row_means)
print("Row variances:", row_variances)

Row means: [ 1.50794299e-18  1.96032588e-17 -1.20635439e-17 -9.42464367e-18
 -1.07440938e-17]
Row variances: [1. 1. 1. 1. 1.]


In [11]:
U, sigma, Vt = np.linalg.svd(Z, full_matrices=False)
Y = U.T @ Z

C_Z = Z @ Z.T / m
eigenvalues, eigenvectors = np.linalg.eigh(C_Z)

idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Singular values:", sigma)
print("Eigenvalues:", eigenvalues)
print("sigma^2 / m:", sigma**2 / m)

Singular values: [122.77186576  65.43572424  49.61861925  37.84859449  17.62701027]
Eigenvalues: [3.19883935 0.90870841 0.52249732 0.30401445 0.06594047]
sigma^2 / m: [3.19883935 0.90870841 0.52249732 0.30401445 0.06594047]


In [12]:
corr_spy_pc1 = np.corrcoef(Y[0, :], Z[0, :])[0, 1]
print(corr_spy_pc1)

-0.9599783546673651


In [13]:
print("PC1-SPY correlation:", np.corrcoef(Y[0, :], Z[0, :])[0, 1])
print("Loadings:")
print(U)

PC1-SPY correlation: -0.9599783546673651
Loadings:
[[-0.53674156  0.05486835  0.03771315  0.34484587 -0.76717469]
 [-0.51837313  0.08458152  0.17539886  0.55047195  0.62478088]
 [ 0.21935476  0.95274663  0.19625333  0.05531466 -0.05081609]
 [-0.42271387  0.2846345  -0.78179637 -0.33628201  0.12651103]
 [ 0.46518198 -0.03305297 -0.56399648  0.67964451 -0.05004537]]


In [14]:
U[:, 0] *= -1
Vt[0, :] *= -1
Y[0, :] *= -1

In [15]:
print("PC1-SPY correlation:", np.corrcoef(Y[0, :], Z[0, :])[0, 1])
print("Loadings:")
print(U)

PC1-SPY correlation: 0.9599783546673651
Loadings:
[[ 0.53674156  0.05486835  0.03771315  0.34484587 -0.76717469]
 [ 0.51837313  0.08458152  0.17539886  0.55047195  0.62478088]
 [-0.21935476  0.95274663  0.19625333  0.05531466 -0.05081609]
 [ 0.42271387  0.2846345  -0.78179637 -0.33628201  0.12651103]
 [-0.46518198 -0.03305297 -0.56399648  0.67964451 -0.05004537]]


In [16]:
U[:, 2] *= -1
Vt[2, :] *= -1
Y[2, :] *= -1

U[:, 4] *= -1
Vt[4, :] *= -1
Y[4, :] *= -1
print(U)

[[ 0.53674156  0.05486835 -0.03771315  0.34484587  0.76717469]
 [ 0.51837313  0.08458152 -0.17539886  0.55047195 -0.62478088]
 [-0.21935476  0.95274663 -0.19625333  0.05531466  0.05081609]
 [ 0.42271387  0.2846345   0.78179637 -0.33628201 -0.12651103]
 [-0.46518198 -0.03305297  0.56399648  0.67964451  0.05004537]]


In [17]:
for k in range(5):
    if np.dot(eigenvectors[:, k], U[:, k]) < 0:
        eigenvectors[:, k] *= -1

In [18]:
print(np.max(np.abs(eigenvectors - U)))

8.881784197001252e-16


In [21]:
print(np.max(np.abs(Z.T @ U - (sigma[:, None] * Vt).T)))

8.415490526658687e-14
